# MultiBench Part 3: MOSI — L2-MCTN

This notebook runs L2-MCTN (Multimodal Cyclic Translation Network, Level 2) on MOSI. It assumes familiarity with Part 1.

Designed to run on **Google Colab**.

In [ ]:
from google.colab import drive
drive.mount("/content/")

## Clone or update MultiBench

In [ ]:
import os

repo_dir = "/content/MyDrive/MultiBench"

if os.path.isdir(os.path.join(repo_dir, ".git")):
    print("Repo already exists, pulling latest changes...")
    !git stash
    !git -C "$repo_dir" pull
else:
    print("Cloning MultiBench...")
    !git clone https://github.com/human-ai-lab/MultiBench.git "$repo_dir"

print(f"Repo location: {repo_dir}")

## Setup Python path

In [ ]:
import sys

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

print(f"Using MultiBench from: {repo_dir}")

## Check and download MOSI data

Data is stored at `MultiBench/data/affect/mosi_raw.pkl` — inside the repo on Drive, so it persists across sessions and matches the path used by the local `.py` scripts.

In [ ]:
data_dir  = os.path.join(repo_dir, "data", "affect")
data_path = os.path.join(data_dir, "mosi_raw.pkl")

os.makedirs(data_dir, exist_ok=True)

if os.path.exists(data_path):
    print(f"Data found: {data_path}")
else:
    print("Downloading mosi_raw.pkl (one-time)...")
    !wget -q --show-progress -O "$data_path" https://filedn.eu/lDTxyzlMbdMJJq0AvECx20X/mosi_raw.pkl
    print(f"Saved to: {data_path}")

print(f"data_path = {data_path}")

## Install dependencies

In [ ]:
!pip install -q memory-profiler

## Setup device

In [ ]:
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load MOSI dataset

In [ ]:
from datasets.affect.get_data import get_dataloader

traindata, validdata, testdata = get_dataloader(data_path, robust_test=False)

## Define encoder and decoder modules

In [ ]:
from unimodals.common_models import GRU, MLP
from fusions.MCTN import Encoder, Decoder

feature_dim = 300
hidden_dim = 32

encoder0 = Encoder(feature_dim, hidden_dim, n_layers=1, dropout=0.0).to(device)
decoder0 = Decoder(hidden_dim, feature_dim, n_layers=1, dropout=0.0).to(device)
encoder1 = Encoder(hidden_dim, hidden_dim, n_layers=1, dropout=0.0).to(device)
decoder1 = Decoder(hidden_dim, feature_dim, n_layers=1, dropout=0.0).to(device)

reg_encoder = nn.GRU(hidden_dim, 32).to(device)

## Define prediction head

In [ ]:
head = MLP(32, 64, 1).to(device)

## Train L2-MCTN

In [ ]:
from private_test_scripts.all_in_one import all_in_one_train
from training_structures.MCTN_Level2 import train, test

allmodules = [encoder0, decoder0, encoder1, decoder1, reg_encoder, head]

def trainprocess():
    train(
        traindata, validdata,
        encoder0, decoder0, encoder1, decoder1,
        reg_encoder, head,
        criterion_t0=nn.MSELoss(), criterion_c=nn.MSELoss(),
        criterion_t1=nn.MSELoss(), criterion_r=nn.L1Loss(),
        max_seq_len=20,
        mu_t0=0.01, mu_c=0.01, mu_t1=0.01,
        dropout_p=0.15, early_stop=False, patience_num=15,
        lr=1e-4, weight_decay=0.01, op_type=torch.optim.AdamW,
        epoch=10, model_save="best_mctn.pt")

all_in_one_train(trainprocess, allmodules)

## Test

In [ ]:
model = torch.load("best_mctn.pt", map_location=device, weights_only=False).to(device)
test(model, testdata, "mosi", no_robust=True)

That concludes Part 3! Feel free to open an issue on GitHub if you have any questions.